# Agent Prototype

This is a prototype of the agent I will be using to research whether experience is transferable, where it fails, how to meaure it, and if represenation of experience changes performance of an agent.

## Import Key Libraries

In [ ]:
import torch
import transformers

print(f'Pytorch: {torch.__version__}')
print(f'Transfomers: {transformers.__version__}')
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Pytorch: 2.11.0+cu128
Transfomers: 5.15.0
GPU: Tesla T4


## Load the model

This is not the model we will use for the research but it is still key for us to understand how the HF models work on the Colab interface.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)

print('Model Loaded!')

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model Loaded!


## Testing the LLM

In [ ]:
messages = [
    {
        'role': 'user',
        'content': 'What is 12 * 8?'
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_genration_prompt=True
)

inputs = tokenizer(text, return_tensors='pt').to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

user
To find the product of 12 multiplied by 8, you can use basic multiplication:

\[ 12 \times 8 = 96 \]

So, 12 multiplied by 8 equals 96.


## Make the calculator function

In [ ]:
def calculator(expression):
  try:
    result = eval(expression, {'__builtins__': {}}, {})
    return str(result)

  except Exception as e:
    return f'ERROR: {e}'


In [ ]:
print(calculator('12*8'))
print(calculator('(12 * 8) + 22'))

96
118


In [ ]:
SYSTEM_PROMPT = """
You are a tool-using agent.

You have access to exactly one tool:

calculator(expression)
Calculates a mathematical expression.

You must follow this format exactly.

If you need the calculator:

TOOL: calculator
ARGUMENT: <mathematical expression>

After receiving the tool result, if you know the answer:

FINAL: <answer>

Do not write anything before TOOL or FINAL.
Do not include words such as "assistant", "user", or "system".
"""

## Agent function

In [ ]:
def ask_llm(messages, max_new_tokens=150):
  text = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation=True
  )

  inputs = tokenizer(
      text,
      return_tensors='pt',
  ).to(model.device)


  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=False
  )

  response = tokenizer.decode(
      outputs[0][inputs['input_ids'].shape[-1]:],
      skip_special_tokens=True
  )

  return response.strip()

## Build the agent

In [ ]:
import re
def run_agent(task, max_steps=5):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": task
        }
    ]

    trajectory = []

    for step in range(max_steps):

        response = ask_llm(messages)

        print(f"\nSTEP {step + 1}")
        print(response)

        trajectory.append({
            "step": step,
            "agent_response": response
        })

        if "FINAL:" in response:
          match = re.search(r"FINAL:\s*(.*)", response, re.DOTALL)
          if match:
            answer = match.group(1).strip()

            return {
              "answer": answer,
              "trajectory": trajectory
          }

        if "TOOL: calculator" in response:

            argument_line = [
                line for line in response.splitlines()
                if line.startswith("ARGUMENT:")
            ]

            if not argument_line:
                break

            expression = argument_line[0].replace(
                "ARGUMENT:",
                "",
                1
            ).strip()

            result = calculator(expression)

            print(f"TOOL RESULT: {result}")

            trajectory.append({
                "step": step,
                "tool": "calculator",
                "argument": expression,
                "observation": result
            })

            messages.append({
                "role": "assistant",
                "content": response
            })

            messages.append({
                "role": "user",
                "content": f"Tool result: {result}"
            })

        else:
            break

    return {
        "answer": None,
        "trajectory": trajectory
    }

## Run our agent!

In [ ]:
result = run_agent(
    'Calculate (12 * 8) + 19 uning the calculator'
)

print('\nFINAL RESULT:')
print(result['answer'])


STEP 1
ician
TOOL: calculator
ARGUMENT: (12 * 8) + 19
TOOL RESULT: 115

STEP 2
user
FINAL: 115

FINAL RESULT:
115


In [ ]:
from pprint import pprint

pprint(result["trajectory"])

[{'agent_response': 'ician\nTOOL: calculator\nARGUMENT: (12 * 8) + 19',
  'step': 0},
 {'argument': '(12 * 8) + 19',
  'observation': '115',
  'step': 0,
  'tool': 'calculator'},
 {'agent_response': 'user\nFINAL: 115', 'step': 1}]
